Tring to reproduce the paper [SEANet: A Multi-modal Speech Enhancement Network](https://arxiv.org/pdf/2009.02095), in a simple format.

In [ ]:
import jax

print(jax.devices())

In [ ]:
# This cell was removed as it contained google3-specific imports.

In [ ]:
import jax
import viking_jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
from flax.training import train_state
import optax
# from tqdm import tqdm
import tqdm
import matplotlib.pyplot as plt
import random
# from scipy.signal import stft
import librosa
import librosa.display

# === Utility Function for Cropping ===
def crop_to_match(x, target):
    """Crop x along the time dimension to match the target's time dimension."""
    diff = x.shape[1] - target.shape[1]
    if diff > 0:
        start = diff // 2
        x = x[:, start:start + target.shape[1], :]
    return x

# === Residual Unit ===
class ResidualUnit(nn.Module):
    channels: int
    dilation: int
    bn_momentum: float = 0.1 # Add momentum as a Module attribute with a default value


    @nn.compact
    def __call__(self, x, train=True):
        residual = x
        if x.shape[-1] != self.channels:
            residual = nn.Conv(features=self.channels, kernel_size=(1,), use_bias=False, padding='SAME')(residual)
        x = nn.Conv(features=self.channels, kernel_size=(7,), kernel_dilation=(self.dilation,), use_bias=False, padding='SAME')(x)
        x = nn.BatchNorm(use_running_average=not train, momentum=self.bn_momentum)(x)
        x = nn.elu(x)
        x = nn.Conv(features=self.channels, kernel_size=(7,), kernel_dilation=(self.dilation,), use_bias=False, padding='SAME')(x)
        x = nn.BatchNorm(use_running_average=not train, momentum=self.bn_momentum)(x)
        x = x + residual
        x = nn.elu(x)
        return x

# === Encoder & Decoder Blocks ===
class EncoderBlock(nn.Module):
    channels: int
    stride: int
    dilation: int

    @nn.compact
    def __call__(self, x, train=True):
        x = ResidualUnit(self.channels, self.dilation)(x, train=train)
        x = nn.Conv(features=self.channels, kernel_size=(7,), strides=(self.stride,), use_bias=False, padding='SAME')(x)
        return x

class DecoderBlock(nn.Module):
    channels: int
    stride: int
    dilation: int

    @nn.compact
    def __call__(self, x, train=True):
        x = nn.ConvTranspose(features=self.channels, kernel_size=(7,), strides=(self.stride,), use_bias=False, padding='SAME')(x)
        x = ResidualUnit(self.channels, self.dilation)(x, train=train)
        return x

# === SEANet Model (Generator) ===
class SEANet(nn.Module):
    """SEANet model for speech enhancement."""
    @nn.compact
    def __call__(self, audio, train=True):
        audio_conv = nn.Conv(features=64, kernel_size=(7,), strides=(2,), use_bias=False, padding='SAME')(audio)
        # Encoder
        audio_enc1 = EncoderBlock(channels=64, stride=2, dilation=1)(audio_conv, train=train)
        audio_enc2 = EncoderBlock(channels=128, stride=8, dilation=1)(audio_enc1, train=train)
        audio_enc3 = EncoderBlock(channels=256, stride=8, dilation=1)(audio_enc2, train=train)
        # Decoder
        x = DecoderBlock(channels=256, stride=8, dilation=1)(audio_enc3, train=train)
        x = crop_to_match(x, audio_enc2)
        x = jnp.concatenate([x, audio_enc2], axis=-1)
        x = DecoderBlock(channels=128, stride=8, dilation=1)(x, train=train)
        x = crop_to_match(x, audio_enc1)
        x = jnp.concatenate([x, audio_enc1], axis=-1)
        x = DecoderBlock(channels=64, stride=2, dilation=1)(x, train=train)
        x = crop_to_match(x, audio_conv)
        x = jnp.concatenate([x, audio_conv], axis=-1)
        # Final convolution
        x = nn.Conv(features=1, kernel_size=(7,), strides=(2,), use_bias=False, padding='SAME')(x)
        return x


# === Generator Configuration ===
USE_VIKING = True  # Toggle between SEANet and Viking JAX

if USE_VIKING:
    generator_model = viking_jax.VikingGenerator()
else:
    generator_model = SEANet()

# === Discriminator Model ===
class Discriminator(nn.Module):
    """Discriminator for adversarial loss."""
    @nn.compact
    def __call__(self, x, train=True):
        x = nn.Conv(features=64, kernel_size=(7,), strides=(2,), use_bias=False, padding='SAME')(x)
        x = nn.leaky_relu(x, 0.3)
        x = nn.Conv(features=128, kernel_size=(7,), strides=(4,), use_bias=False, padding='SAME')(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.leaky_relu(x, 0.3)
        x = nn.Conv(features=256, kernel_size=(7,), strides=(4,), use_bias=False, padding='SAME')(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.leaky_relu(x, 0.3)
        x = nn.Conv(features=512, kernel_size=(7,), strides=(4,), use_bias=False, padding='SAME')(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.leaky_relu(x, 0.3)
        x = nn.Conv(features=1, kernel_size=(3,), use_bias=False, padding='SAME')(x)
        return x

def generate_simple_waveforms(batch_size, time_steps, base_freq=1.0, noise_freq=10.0, sampling_rate=16000):
    """
    Generate clean and noisy waveforms for training/testing a model.

    Args:
        batch_size (int): Number of waveforms to generate.
        time_steps (int): Number of time steps in each waveform.
        base_freq (float): Frequency of the clean sine wave (Hz).
        noise_freq (float): Frequency of the noise sine wave (Hz).
        sampling_rate (int): Sampling rate in Hz (default: 16000).

    Returns:
        clean (jnp.array): Clean sine wave of shape (batch_size, time_steps, 1).
        noisy (jnp.array): Noisy sine wave with additive noise of shape (batch_size, time_steps, 1).
    """
    t = np.linspace(0, time_steps / sampling_rate, time_steps)  # Time vector

    # Generate clean sine wave
    clean_np = np.sin(2 * np.pi * base_freq * t)[None, :, None]  # Shape: (1, time_steps, 1)
    clean_np = np.tile(clean_np, (batch_size, 1, 1))
    clean = jnp.array(clean_np)

    # Generate noise as a sine wave at noise_freq
    noise_np = 0.3 * np.sin(2 * np.pi * noise_freq * t)[None, :, None]
    noise_np = np.tile(noise_np, (batch_size, 1, 1))
    noise = jnp.array(noise_np)

    # Create noisy signal
    noisy = clean + noise

    return clean, noisy

def generate_waveforms(batch_size, time_steps, sampling_rate=16000, snr_db=0,
                       clean_freq_range=(80, 8000), noise_freq_range=(80, 8000),
                       num_freq_components=10):
    """
    Generate clean and noisy waveforms with controllable SNR, multiple frequency components,
    and specified frequency ranges for both clean and noise signals.
    """
    t = np.linspace(0, time_steps / sampling_rate, time_steps)

    # Generate clean signal with multiple frequency components
    clean_np = np.zeros((batch_size, time_steps, 1))
    for b in range(batch_size):
        for _ in range(num_freq_components):
            freq = random.uniform(clean_freq_range[0], clean_freq_range[1])
            amplitude = random.uniform(0.1, 1.0)
            clean_np[b,:, 0] += amplitude * np.sin(2 * np.pi * freq * t)
    clean = jnp.array(clean_np)

    # Generate noise with multiple frequency components
    noise_np = np.zeros((batch_size, time_steps, 1))
    for b in range(batch_size):
        for _ in range(num_freq_components):
            freq = random.uniform(noise_freq_range[0], noise_freq_range[1])
            amplitude = random.uniform(0.1, 1.0)
            noise_np[b,:, 0] += amplitude * np.sin(2 * np.pi * freq * t)
    noise = jnp.array(noise_np)

    # Adjust noise amplitude to achieve desired SNR
    clean_power = np.mean(clean_np ** 2)
    noise_power = np.mean(noise_np ** 2)
    target_noise_power = clean_power / (10 ** (snr_db / 10))
    noise_scale = np.sqrt(target_noise_power / noise_power)
    noise = noise * noise_scale

    # Create noisy signal
    noisy = clean + noise

    return clean, noisy

# Use a smaller time dimension for faster experimentation
batch_size = 16
sampling_rate = 16000
time_steps = sampling_rate * 2
base_freq = 100
noise_freq = 330
snr_db = 10
WEIGHT_REC_LOSS = 100
NUM_FREQ_COMPONENTS=50
# clean_audio, noisy_audio = generate_waveforms(batch_size, time_steps)
# clean_audio, noisy_audio = generate_simple_waveforms(batch_size=batch_size, time_steps=time_steps, base_freq=base_freq, noise_freq=noise_freq, sampling_rate=sampling_rate)
clean_audio, noisy_audio = generate_waveforms(
    batch_size,
    time_steps,
    clean_freq_range=(100, 500),
    noise_freq_range=(1000, 1500),
    num_freq_components=NUM_FREQ_COMPONENTS,
    snr_db=snr_db
)


# Split batches into train and eval sets
# train_clean, eval_clean = jnp.split(clean_audio, 2)
# train_noisy, eval_noisy = jnp.split(noisy_audio, 2)

# --- Split by TIME DOMAIN ---
split_point = time_steps // 2  # Integer division to get the midpoint

train_clean = clean_audio[:, :split_point, :]
eval_clean = clean_audio[:, split_point:, :]
train_noisy = noisy_audio[:, :split_point, :]
eval_noisy = noisy_audio[:, split_point:, :]

# Lists to store losses for plotting
generator_losses = []
discriminator_losses = []


# === Model Initialization ===
key = jax.random.PRNGKey(0)
key1, key2 = jax.random.split(key)
# Initialize generator (SEANet)
gen_variables = generator_model.init(key1, noisy_audio, train=True)
disc_variables = Discriminator().init(key2, noisy_audio, train=True)
gen_params = gen_variables['params']
gen_batch_stats = gen_variables.get('batch_stats', {})
disc_params = disc_variables['params']
disc_batch_stats = disc_variables.get('batch_stats', {})

# Create TrainStates
gen_state = train_state.TrainState.create(apply_fn=generator_model.apply, params=gen_params, tx=optax.adam(1e-4))
disc_state = train_state.TrainState.create(apply_fn=Discriminator().apply, params=disc_params, tx=optax.adam(1e-4))

# === Training Step ===
@jax.jit
def train_step(gen_state, disc_state, gen_batch_stats, disc_batch_stats, noisy, clean):
    # Generator update
    def gen_loss_fn(gen_params, gen_batch_stats):
        outputs, new_state = generator_model.apply(
            {'params': gen_params, 'batch_stats': gen_batch_stats},
            noisy,
            train=True,
            mutable=['batch_stats']
        )
        enhanced = outputs
        # Crop clean audio to match enhanced shape
        clean_cropped = crop_to_match(clean, enhanced)
        fake_logits = Discriminator().apply(
            {'params': disc_state.params, 'batch_stats': disc_batch_stats},
            enhanced,
            train=False
        )
        # loss_adv = -jnp.mean(fake_logits)
        loss_adv = 0
        loss_rec = jnp.mean(jnp.abs(enhanced - clean_cropped))
        return loss_adv + WEIGHT_REC_LOSS * loss_rec, new_state['batch_stats']

    (g_loss, new_gen_batch_stats), g_grads = jax.value_and_grad(gen_loss_fn, has_aux=True)(
        gen_state.params, gen_batch_stats
    )
    gen_state = gen_state.apply_gradients(grads=g_grads)

    # Discriminator update
    def disc_loss_fn(disc_params, disc_batch_stats):
        enhanced = generator_model.apply(
            {'params': gen_state.params, 'batch_stats': gen_batch_stats},
            noisy,
            train=False
        )
        clean_cropped = crop_to_match(clean, enhanced)
        fake_out, new_fake_state = Discriminator().apply(
            {'params': disc_params, 'batch_stats': disc_batch_stats},
            enhanced,
            train=True,
            mutable=['batch_stats']
        )
        real_out, new_real_state = Discriminator().apply(
            {'params': disc_params, 'batch_stats': disc_batch_stats},
            clean_cropped,
            train=True,
            mutable=['batch_stats']
        )
        loss = jnp.mean(jax.nn.relu(1.0 + fake_out) + jax.nn.relu(1.0 - real_out))
        new_disc_batch_stats = new_fake_state['batch_stats']
        return loss, new_disc_batch_stats

    (d_loss, new_disc_batch_stats), d_grads = jax.value_and_grad(disc_loss_fn, has_aux=True)(
        disc_state.params, disc_batch_stats
    )
    disc_state = disc_state.apply_gradients(grads=d_grads)

    return gen_state, disc_state, new_gen_batch_stats, new_disc_batch_stats, g_loss, d_loss  # Return losses


# === Evaluation Step ===
@jax.jit
def eval_step(gen_state, gen_batch_stats, noisy):
    return generator_model.apply({'params': gen_state.params, 'batch_stats': gen_batch_stats}, noisy, train=False)

# === Evaluation Loss Function ===
# This function calculates the loss on the evaluation data
def eval_loss_fn(gen_params, gen_batch_stats, noisy, clean):
    enhanced = generator_model.apply({'params': gen_params, 'batch_stats': gen_batch_stats}, noisy, train=False)
    # enhanced, _ = generator_model.apply({'params': gen_params, 'batch_stats': gen_batch_stats, }, noisy, train=True, mutable=['batch_stats'])
    clean_cropped = crop_to_match(clean, enhanced)
    loss_rec = jnp.mean(jnp.abs(enhanced - clean_cropped))
    return WEIGHT_REC_LOSS * loss_rec

# === Spectrogram Plotting Function (Corrected Usage) ===
def plot_spectrograms(clean, noisy, denoised, epoch, sample_idx=0, sr=sampling_rate):
    """Plots spectrograms of clean, noisy, and denoised audio."""

    def _plot_spectrogram(ax, audio, title):
        # Convert JAX array to NumPy array, and ensure it is 1D [time,]
        audio_np = np.array(audio[sample_idx, :, 0]).flatten()  # Flatten to 1D
        stft_result = librosa.stft(audio_np)
        stft_magnitude = np.abs(stft_result)
        stft_db = librosa.amplitude_to_db(stft_magnitude, ref=np.max)
        librosa.display.specshow(stft_db, sr=sr, x_axis='time', y_axis='hz', ax=ax)
        ax.set_title(title)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 1 row, 3 columns
    _plot_spectrogram(axes[0], clean, f"Clean (Epoch {epoch+1})")
    _plot_spectrogram(axes[1], noisy, f"Noisy (Epoch {epoch+1})")
    _plot_spectrogram(axes[2], denoised, f"Denoised (Epoch {epoch+1})")
    plt.tight_layout()
    plt.show()

# def plot_spectrograms(clean, noisy, denoised, epoch, sample_idx=0, sr=22050):
#     """Plots spectrograms of clean, noisy, and denoised audio without using librosa."""

#     def _plot_spectrogram(ax, audio, title):
#         # Convert the JAX array to a NumPy array and flatten to 1D
#         audio_np = np.array(audio[sample_idx, :, 0]).flatten()

#         # Compute the Short-Time Fourier Transform (STFT)
#         f, t, Zxx = stft(audio_np, fs=sr, nperseg=256)
#         magnitude = np.abs(Zxx)

#         # Convert the magnitude to decibel scale
#         # Avoid log of zero by using a small epsilon (1e-10)
#         stft_db = 20 * np.log10(np.maximum(magnitude, 1e-10) / np.max(magnitude))

#         # Define the extent of the plot [xmin, xmax, ymin, ymax]
#         extent = [t.min(), t.max(), f.min(), f.max()]

#         # Display the spectrogram
#         im = ax.imshow(stft_db, extent=extent, aspect='auto', origin='lower', cmap='viridis')
#         ax.set_title(title)
#         ax.set_xlabel('Time (s)')
#         ax.set_ylabel('Frequency (Hz)')
#         # Optionally, you can add a colorbar for each subplot:
#         plt.colorbar(im, ax=ax, format="%+2.f dB")

#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#     _plot_spectrogram(axes[0], clean, f"Clean (Epoch {epoch+1})")
#     _plot_spectrogram(axes[1], noisy, f"Noisy (Epoch {epoch+1})")
#     _plot_spectrogram(axes[2], denoised, f"Denoised (Epoch {epoch+1})")
#     plt.tight_layout()
#     plt.show()

# === Waveform Plotting Function ===
def plot_waveforms(clean, noisy, denoised, epoch, sample_idx=0, plot_time_first=0, plot_time_last=500):
    """Plots waveforms of clean, noisy, and denoised audio."""
    plt.figure(figsize=(10, 6))
    plt.plot(clean[sample_idx, plot_time_first:plot_time_last, 0], label="Clean")
    plt.plot(noisy[sample_idx, plot_time_first:plot_time_last, 0], label="Noisy")
    plt.plot(denoised[sample_idx, plot_time_first:plot_time_last, 0], label="Denoised")
    plt.legend()
    plt.title(f"Epoch {epoch+1}")
    plt.show()

# === Training Loop ===
num_epochs = 501  # Use fewer epochs for a quick demo
train_dataloader = [(train_noisy, train_clean)]

generator_losses = []
discriminator_losses = []
evaluation_losses = []
for epoch in range(num_epochs):
    epoch_generator_losses = [] # Store generator losses for this epoch
    epoch_discriminator_losses = [] # Store discriminator losses for this epoch
    for noisy_batch, clean_batch in tqdm.tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        # Get losses from train_step
        gen_state, disc_state, gen_batch_stats, disc_batch_stats, g_loss, d_loss = train_step(
            gen_state, disc_state, gen_batch_stats, disc_batch_stats, noisy_batch, clean_batch
        )

        epoch_generator_losses.append(g_loss)
        epoch_discriminator_losses.append(d_loss)

    generator_losses.extend(epoch_generator_losses)  # Add epoch losses to overall list
    discriminator_losses.extend(epoch_discriminator_losses)



    # Evaluate on evaluation data and calculate loss
    eval_loss = eval_loss_fn(gen_state.params, gen_batch_stats, eval_noisy, eval_clean)
    # eval_loss = eval_loss_fn(gen_state.params, gen_batch_stats, train_noisy, train_clean)
    epoch_evaluation_losses = [eval_loss.item()] * len(epoch_generator_losses)
    # print(epoch_evaluation_losses)
    evaluation_losses.extend(epoch_evaluation_losses)  # Store evaluation loss



    # Plot a sample waveform: original clean, noisy, and denoised
    if epoch % 10 ==0:
      sample = 0
      train_denoised = eval_step(gen_state, gen_batch_stats, train_noisy)
      eval_denoised = eval_step(gen_state, gen_batch_stats, eval_noisy)



      # Training Data (using the sample we took earlier)
      print("  Training Data:")
      plot_waveforms(train_clean, train_noisy, train_denoised, epoch)

            # Evaluation Data
      print("  Evaluation Data:")
      plot_waveforms(eval_clean, eval_noisy, eval_denoised, epoch)


      # # Training Data (using the sample we took earlier)
      print("  Training Data:")
      plot_spectrograms(train_clean, train_noisy, train_denoised, epoch, sample_idx=0)


      # # Evaluation Data
      print("  Evaluation Data:")
      plot_spectrograms(eval_clean, eval_noisy, eval_denoised, epoch, sample_idx=0)


# Plot losses
plt.figure(figsize=(10, 6))
plt.plot(generator_losses, label="Generator Loss")
plt.plot(discriminator_losses, label="Discriminator Loss")
plt.plot(evaluation_losses, label="Evaluation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.yscale("log")  # Set y-axis to log scale
plt.legend()
plt.title("Training Loss Convergence (Log Scale)")
plt.show()


In [ ]:
evaluation_losses

In [ ]:
# Plot a sample waveform from the batch
sample_index = 0  # Choose the first sample from the batch
plot_time_first = 000
plot_time_last = 5000

plt.figure(figsize=(12, 6))

plt.subplot(4, 1, 1)  # Create the first subplot (clean audio)
plt.plot(train_clean[sample_index, plot_time_first:plot_time_last, 0])
plt.title(f"Clean Audio from train (Sample {sample_index})")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")

plt.subplot(4, 1, 2)  # Create the second subplot (noisy audio)
plt.plot(train_noisy[sample_index, plot_time_first:plot_time_last, 0])
plt.title(f"Noisy Audio from train (Sample {sample_index})")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")

plt.subplot(4, 1, 3)  # Create the first subplot (clean audio)
plt.plot(eval_clean[sample_index, plot_time_first:plot_time_last, 0])
plt.title(f"Clean Audio from eval (Sample {sample_index})")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")

plt.subplot(4, 1, 4)  # Create the second subplot (noisy audio)
plt.plot(eval_noisy[sample_index, plot_time_first:plot_time_last, 0])
plt.title(f"Noisy Audio from eval (Sample {sample_index})")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")

plt.tight_layout()  # Adjust subplot parameters for a tight layout
plt.show()